# 01 — Data Exploration
## Cedar Creek Fire, Eugene OR — Aug–Sep 2022

This notebook loads all three raw data sources, checks coverage, and produces
an initial visual overview before any cleaning or merging.

**Sources:**
- PurpleAir — 57 community sensors, 60-min averages
- NOAA ASOS KEUG — sub-hourly METAR observations, Eugene airport
- LRAPA — hourly regulatory PM2.5, 7 Lane County stations

In [ ]:
import sys
sys.path.append("../src")

from data_loader import PurpleAirLoader, NOAALoader, LRAPALoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 1. PurpleAir Data

In [ ]:
pa_loader = PurpleAirLoader(data_dir="../data/raw/purpleair")
pa_raw = pa_loader.load_all_sensors_in_directory()

print(f"\nShape:    {pa_raw.shape}")
print(f"Sensors:  {pa_raw['sensor_id'].nunique()}")
print(f"Columns:  {pa_raw.columns.tolist()}")
print(f"Date range: {pa_raw['timestamp'].min()} → {pa_raw['timestamp'].max()}")
pa_raw.head(3)

In [ ]:
# Per-sensor coverage: how many hourly readings does each sensor have?
expected_hours = 61 * 24  # Aug 1 – Sep 30
sensor_counts = pa_raw.groupby('sensor_id')['timestamp'].count().sort_values()
coverage_pct = (sensor_counts / expected_hours * 100).round(1)

print(f"Expected readings per sensor: {expected_hours}")
print(f"\nCoverage (%):")
print(coverage_pct.to_string())
print(f"\nMedian coverage: {coverage_pct.median():.1f}%")
print(f"Sensors with >90% coverage: {(coverage_pct > 90).sum()}")

In [ ]:
# Raw PM2.5 time series — all sensors (gray) + median (red)
fig, ax = plt.subplots(figsize=(14, 5))

for sid, grp in pa_raw.groupby('sensor_id'):
    ax.plot(grp['timestamp'], grp['pm2.5_cf_1_a'],
            alpha=0.12, color='steelblue', linewidth=0.6)

median_ts = pa_raw.groupby('timestamp')['pm2.5_cf_1_a'].median()
ax.plot(median_ts.index, median_ts.values,
        color='red', linewidth=1.8, label='Median across sensors', zorder=5)

ax.axhline(35, color='orange', linestyle='--', linewidth=1, alpha=0.8, label='EPA 24h standard (35 µg/m³)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_ylabel('PM2.5 CF=1 (µg/m³)')
ax.set_title('PurpleAir Raw PM2.5 — All 57 Sensors, Eugene OR (Aug–Sep 2022)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig_pa_raw_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. NOAA Weather Data (KEUG — Eugene Airport)

In [ ]:
noaa_loader = NOAALoader(data_dir="../data/raw/noaa")
noaa_raw = noaa_loader.load_all_weather_data()

# Keep only the EUG station (Eugene airport ASOS); drop the 77S auxiliary station
noaa_eug = noaa_raw[noaa_raw['station'] == 'EUG'].copy()
print(f"NOAA raw records (all stations): {len(noaa_raw)}")
print(f"NOAA EUG records:                {len(noaa_eug)}")
print(f"Date range: {noaa_eug['timestamp'].min()} → {noaa_eug['timestamp'].max()}")
print(f"\nWeather columns available: {[c for c in noaa_eug.columns if c not in ['station','valid','timestamp','metar']]}")
noaa_eug[['timestamp','temperature_f','humidity','wind_speed_mph','wind_direction','pressure_hpa']].head(5)

In [ ]:
# NOAA weather overview — four-panel time series
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
weather_vars = [
    ('temperature_f', 'Temperature (°F)', 'tomato'),
    ('humidity',      'Relative Humidity (%)', 'steelblue'),
    ('wind_speed_mph','Wind Speed (mph)', 'seagreen'),
    ('pressure_hpa',  'Pressure (hPa)', 'mediumpurple'),
]
for ax, (col, label, color) in zip(axes, weather_vars):
    if col in noaa_eug.columns:
        ax.plot(noaa_eug['timestamp'], noaa_eug[col], color=color, linewidth=0.7, alpha=0.8)
        ax.set_ylabel(label)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[0].set_title('NOAA KEUG Weather — Eugene Airport (Aug–Sep 2022)')
plt.tight_layout()
plt.savefig('../data/processed/fig_noaa_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. LRAPA Regulatory Data

In [ ]:
lrapa_loader = LRAPALoader(data_dir="../data/raw/lrapa")
lrapa_raw = lrapa_loader.load_all_lrapa_data()
print(lrapa_loader.get_data_summary(lrapa_raw))
lrapa_raw.head(3)

In [ ]:
# LRAPA all stations + regulatory mean
station_cols = ['pm2.5_amazon_park', 'pm2.5_highway_99', 'pm2.5_santa_clara', 'pm2.5_springfield']
station_cols = [c for c in station_cols if c in lrapa_raw.columns]

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for col, color in zip(station_cols, colors):
    ax.plot(lrapa_raw['timestamp'], lrapa_raw[col],
            alpha=0.5, linewidth=0.8, color=color, label=col.replace('pm2.5_', '').replace('_', ' ').title())

ax.plot(lrapa_raw['timestamp'], lrapa_raw['pm2.5_lrapa_regulatory'],
        color='black', linewidth=2, label='Eugene-area mean (regulatory)', zorder=5)
ax.axhline(35, color='orange', linestyle='--', linewidth=1, alpha=0.8, label='EPA standard')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('LRAPA Regulatory PM2.5 — Lane County Stations (Aug–Sep 2022)')
ax.legend(fontsize=9, ncol=3)
plt.tight_layout()
plt.savefig('../data/processed/fig_lrapa_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Data Summary

In [ ]:
print("=== Missing Data Summary ===")
print(f"\nPurpleAir (per key column):")
for col in ['pm2.5_cf_1_a', 'pm2.5_cf_1_b', 'pm2.5_alt_a', 'humidity_a']:
    if col in pa_raw.columns:
        pct = pa_raw[col].isna().mean() * 100
        print(f"  {col:25s}: {pct:.1f}% missing")

print(f"\nNOAA EUG (per key column):")
for col in ['temperature_f', 'humidity', 'wind_speed_mph', 'pressure_hpa']:
    if col in noaa_eug.columns:
        pct = noaa_eug[col].isna().mean() * 100
        print(f"  {col:25s}: {pct:.1f}% missing")

print(f"\nLRAPA regulatory:")
pct = lrapa_raw['pm2.5_lrapa_regulatory'].isna().mean() * 100
print(f"  pm2.5_lrapa_regulatory  : {pct:.1f}% missing")